In [1]:
import math
import random
import numpy as np

from datetime import datetime
from pyomo.environ import *

import pandapower as pp
import pandapower.networks as pn
import pandapower.converter as pc
from pandapower.pypower.makeBdc import makeBdc
from pandapower.pypower.idx_gen import PG, GEN_BUS, PMAX, PMIN
# from pandapower.pypower.idx_cost import NCOST, COST
from pandapower.pypower.idx_brch import PF, PT, QF, QT, RATE_A, F_BUS, T_BUS
from pandapower.pypower.idx_bus import PD

from pypower.api import runpf, ppoption

from clmp_phq import CLMP
from clmp_phq import find_all_branches
from pandapower.pypower.makePTDF import makePTDF
from pyomo.environ import value

In [11]:
path = 'C:\\Users\\RichriD\\Downloads\\Compressed\\Ipopt-3.14.17-win64-msvs2022-md\\bin\\ipopt.exe'
case = 'case9'
net = pn.case9()
e_g = [0.3, 0.5, 0.8]

run_id = case + '-' + datetime.now().strftime("%Y%m%d") + '-' + str(random.randint(0, 9999))
structre_path = 'res/model_structure-' + run_id + '.txt'
variable_path = 'res/model_variable-' + run_id + '.txt'
constraints_path = 'res/constraints-' + run_id + '.txt'
path_list = [structre_path, variable_path, constraints_path]

clmp = CLMP(net, e_g, solver_name='ipopt', verbose=True, path=path_list)
# clmp = CLMP(net, e_g, solver_name='ipopt', verbose=True)
clmp.create_model()
_ = clmp.opt_solve()

Ipopt 3.14.17: tol=1e-06
acceptable_tol=1e-05


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.17, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:      151
Number of nonzeros in inequality constraint Jacobian.:      117
Number of nonzeros in Lagrangian Hessian.............:       51

Total number of variables............................:       53
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       14
                     variables with only upper bounds:        0
Total number of equality constraints.........

In [13]:
clmp.export_results()

# 目标节点假定为 6

In [4]:
n = 6

In [5]:
# Note 几个重要的集合(矩阵)
# 1. 节点的邻接矩阵
# 2. 节点-上下游支路的邻接矩阵 (实际是两个, 分别对应上下游的情况)
# 上游节点集合
p_in_result = np.zeros((clmp.bus_num, clmp.bus_num))
# 遍历变量并赋值
for (i, j) in clmp.model.p_in:
    if (i in clmp.model.buses) and (j in clmp.model.buses):
        p_in_result[i, j] = np.round(value(clmp.model.p_in[i, j]), 3)
p_in_set = (p_in_result > 0).astype(int)

In [16]:
p_in_set

array([[0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 1, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 1, 0]])

In [17]:
upper_set = []
for i in range(clmp.bus_num):
    upper_set.append(set(np.where(p_in_set[i, :] == 1)[0]))

In [6]:
# 下游节点集合
p_out_result = np.zeros((clmp.bus_num, clmp.bus_num))
# 遍历变量并赋值
for (i, j) in clmp.model.p_out:
    if (i in clmp.model.buses) and (j in clmp.model.buses):
        p_out_result[i, j] = np.round(value(clmp.model.p_out[i, j]), 3)
p_out_set = (p_out_result > 0).astype(int)

In [7]:
def find_branch_index(branch, from_bus, to_bus):
    for i, br in enumerate(branch):
        if (int(br[F_BUS]) == from_bus and int(br[T_BUS]) == to_bus) or \
           (int(br[F_BUS]) == to_bus and int(br[T_BUS]) == from_bus):
            return i
    return None

In [8]:
# 定义 3-dimension PTDF 矩阵
T_3d = np.zeros((clmp.bus_num, clmp.bus_num, clmp.bus_num))
# 原 T 矩阵的组织方式是按照 branch 中的参考方向组织的, 这里相当于用节点对的形式替换了直接对支路的索引, 增加了一个维度
# 与原参考方向相同的对应元素与 T 中一直, 否则取负值
for l in range(len(clmp.line_list)):
    f_bus = clmp.line_list[l][0]
    t_bus = clmp.line_list[l][1]
    for i in range(clmp.bus_num):
        T_3d[f_bus, t_bus, i] = clmp.T[l, i]
        T_3d[t_bus, f_bus, i] = -clmp.T[l, i]

In [9]:
# lambda_1 对应能量平衡约束
lam1 = [clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.power_balance_constraints.values()]

In [10]:
lam2_for = [clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.forward_flow_eq.values()]
lam2_back = [clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.backward_flow_eq.values()]

In [13]:
# part 1 对于当前节点 n, j 为所有相邻节点
lmp_1_self = -lam1[n] * (-1 + sum(T_3d[n, j, n]
    for j in clmp.adj_set[n]
))

In [14]:
# part 2 对于其他节点
lmp_1_mutal = 0
for k in range(clmp.bus_num):
    if k != n:
        lmp_1_mutal += -lam1[k] *  sum(T_3d[k, j, n]
            for j in clmp.adj_set[k]
        )

In [10]:
# lmp_eb = sum(lam1[i] * T_3d[i, j, n] for i in range(clmp.bus_num) for j in clmp.adj_set[i]) - lam1[n]

In [ ]:
# sum(lam1[i] * T_3d[i, j, n] for i in range(clmp.bus_num) for j in clmp.adj_set[i])

In [15]:
# for i in range(clmp.bus_num):
#     for j in clmp.adj_set[i]:
#         print(i, j)

0 3
1 7
2 5
3 0
3 8
3 4
4 3
4 5
5 2
5 4
5 6
6 5
6 7
7 8
7 1
7 6
8 3
8 7


In [ ]:
[clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.forward_flow_eq.values()]

In [ ]:
[clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.backward_flow_eq.values()]

In [19]:
carbon = [clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.carbon_emission_flow.values()]

In [ ]:
clmp.model.w[0].value

In [15]:
clmp.adj_set

[{3}, {7}, {5}, {0, 4, 8}, {3, 5}, {2, 4, 6}, {5, 7}, {1, 6, 8}, {3, 7}]

In [20]:
sum(carbon[i] * (clmp.model.w[i].value - sum(clmp.model.w[j].value for j in upper_set[i])) * sum(T_3d[i, j, n] for j in upper_set[i]) for i in range(clmp.bus_num))

-1.640090178386384e-05

In [ ]:
[clmp.model.dual[c] if c in clmp.model.dual else None for c in clmp.model.carbon_cap.values()]

In [14]:
T_3d[:, :, 6]

array([[ 0.        ,  0.        ,  0.        , -1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 1.        ,  0.        ,  0.        ,  0.        , -0.46709753,
         0.        ,  0.        ,  0.        , -0.53290247],
       [ 0.        ,  0.        ,  0.        ,  0.46709753,  0.        ,
        -0.46709753,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.        ,  0.        ,  0.46709753,
         0.        , -0.46709753,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.46709753,  0.        ,  0.53290247,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0